In [1]:
# Library Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, linregress
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_validate, cross_val_predict
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, precision_score, recall_score

# Visualization Configuration
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11
plt.rcParams['font.family'] = ['DejaVu Sans', 'Arial', 'sans-serif']

## 1. Configuration and Global Parameters

In [ ]:
# Analysis Parameters
IMPEDANCE_FEATURES = ['ChangeRate_Z(%)', 'ChangeRate_Real(%)', 'ChangeRate_Imag(%)']
FREQUENCIES = [10, 100, 1000, 10000, 100000, 1000000]  # Hz
ABNV_ORDER = ['NC', 'Monomer', 'Trimer', 'Nonamer']  # Biological order
RANDOM_STATE = 42
CV_FOLDS = 5

# Set random seed for reproducibility
np.random.seed(RANDOM_STATE)

# Scoring Weights (Linearity + Biological Order + Classification)
SCORE_WEIGHTS = {
    'linearity': 0.33,
    'order': 0.34,
    'classification': 0.33
}

# Color Palette
ABNV_COLORS = {
    'NC': '#1f77b4',
    'Monomer': '#ff7f0e',
    'Trimer': '#2ca02c',
    'Nonamer': '#d62728'
}

FEATURE_COLORS = {
    'Z': '#3498db',
    'Real': '#e74c3c',
    'Imag': '#2ecc71'
}

In [ ]:
# Machine Learning Models Configuration
ML_MODELS = {
    'ExtraTrees': {
        'model': ExtraTreesClassifier,
        'params': {'n_estimators': 100, 'random_state': RANDOM_STATE}
    },
    'RandomForest': {
        'model': RandomForestClassifier,
        'params': {'n_estimators': 100, 'random_state': RANDOM_STATE}
    },
    'GradientBoosting': {
        'model': GradientBoostingClassifier,
        'params': {'n_estimators': 100, 'random_state': RANDOM_STATE}
    },
    'LogisticRegression': {
        'model': LogisticRegression,
        'params': {'random_state': RANDOM_STATE, 'max_iter': 1000}
    },
    'SVM': {
        'model': SVC,
        'params': {'probability': True, 'random_state': RANDOM_STATE}
    },
    'KNN': {
        'model': KNeighborsClassifier,
        'params': {'n_neighbors': 5}
    }
}

## 2. Utility Functions

In [ ]:
def format_frequency_name(freq):
    """Convert frequency to readable format"""
    if freq >= 1000000:
        return f"{freq//1000000}MHz"
    elif freq >= 1000:
        return f"{freq//1000}kHz"
    else:
        return f"{freq}Hz"

def calculate_linearity_score(features_by_type, abnv_order):
    """Calculate concentration-response linearity using R² from semilog regression"""
    linearity_scores = []
    concentrations = [1e4, 1e5, 1e6, 1e7, 1e8, 1e9]
    log_concentrations = [np.log10(c) for c in concentrations]
    
    for sample_type in abnv_order:
        if sample_type in features_by_type and len(features_by_type[sample_type]) > 0:
            type_data = features_by_type[sample_type]
            data_length = len(type_data)
            
            if data_length >= 3:
                if data_length <= len(log_concentrations):
                    used_log_conc = log_concentrations[:data_length]
                else:
                    used_log_conc = np.linspace(4, 9, data_length)
                
                try:
                    slope, intercept, r_value, p_value, std_err = linregress(used_log_conc, type_data)
                    r_squared = r_value ** 2
                    
                    if 0 <= r_squared <= 1 and not np.isnan(r_squared):
                        linearity_scores.append(r_squared)
                    else:
                        linearity_scores.append(0.0)
                except:
                    linearity_scores.append(0.0)
            else:
                linearity_scores.append(0.0)
        else:
            linearity_scores.append(0.0)
    
    return np.mean(linearity_scores) if linearity_scores else 0.0

def calculate_order_score(features_by_type, abnv_order):
    """Calculate biological order score using Spearman correlation"""
    ordered_means = []
    for sample_type in abnv_order:
        if sample_type in features_by_type and len(features_by_type[sample_type]) > 0:
            ordered_means.append(np.mean(np.abs(features_by_type[sample_type])))
    
    if len(ordered_means) < 3:
        return 0.0
    
    biological_order = [0, 1, 3, 9][:len(ordered_means)]
    
    try:
        order_corr, _ = spearmanr(biological_order, ordered_means)
        
        if np.isnan(order_corr) or order_corr <= 0:
            return 0.0
        elif order_corr < 0.3:
            return 0.1
        else:
            monotonic_increase = all(ordered_means[i] <= ordered_means[i+1] 
                                   for i in range(len(ordered_means)-1))
            return min(order_corr + (0.1 if monotonic_increase else 0.0), 1.0)
    except:
        return 0.0

def calculate_comprehensive_score(linearity, order, classification):
    """Calculate weighted comprehensive score"""
    return (linearity * SCORE_WEIGHTS['linearity'] + 
            order * SCORE_WEIGHTS['order'] + 
            classification * SCORE_WEIGHTS['classification'])

In [ ]:
def calculate_concentration_aware_classification(abnv_data, feature, model_config):
    """Perform concentration-wise classification and aggregate results"""
    concentration_scores = []
    concentration_weights = []
    accuracy_stds = []
    f1_stds = []
    all_true_labels = []
    all_pred_labels = []
    
    for concentration in sorted(abnv_data['Concentration'].unique()):
        conc_data = abnv_data[abnv_data['Concentration'] == concentration]
        
        X_conc = []
        y_conc = []
        
        for sample_type in ABNV_ORDER:
            type_data = conc_data[conc_data['SampleType'] == sample_type]
            if len(type_data) > 0:
                X_conc.extend(type_data[feature].values)
                y_conc.extend([sample_type] * len(type_data))
        
        if len(X_conc) >= 6 and len(np.unique(y_conc)) >= 3:
            X_conc = np.array(X_conc).reshape(-1, 1)
            y_conc = np.array(y_conc)
            
            try:
                model = model_config['model'](**model_config['params'])
                cv_scores = cross_validate(model, X_conc, y_conc, cv=3,
                                         scoring=['accuracy', 'f1_weighted'],
                                         return_train_score=False)
                
                acc_mean = cv_scores['test_accuracy'].mean()
                acc_std = cv_scores['test_accuracy'].std()
                f1_mean = cv_scores['test_f1_weighted'].mean()
                f1_std = cv_scores['test_f1_weighted'].std()
                
                conc_score = (acc_mean + f1_mean) / 2
                
                concentration_scores.append(conc_score)
                concentration_weights.append(len(X_conc))
                accuracy_stds.append(acc_std)
                f1_stds.append(f1_std)
                
                y_pred = cross_val_predict(model, X_conc, y_conc, cv=3)
                all_true_labels.extend(y_conc)
                all_pred_labels.extend(y_pred)
            except:
                continue
    
    if concentration_scores:
        weighted_score = np.average(concentration_scores, weights=concentration_weights)
        weighted_acc_std = np.average(accuracy_stds, weights=concentration_weights) if accuracy_stds else 0
        weighted_f1_std = np.average(f1_stds, weights=concentration_weights) if f1_stds else 0
        
        return weighted_score, all_true_labels, all_pred_labels, weighted_acc_std, weighted_f1_std
    else:
        return 0, [], [], 0, 0

def create_confusion_matrix(abnv_data, feature, model_config):
    """Generate confusion matrix from concentration-wise classification"""
    all_true_labels = []
    all_pred_labels = []
    
    for concentration in sorted(abnv_data['Concentration'].unique()):
        conc_data = abnv_data[abnv_data['Concentration'] == concentration]
        
        X_conc = []
        y_conc = []
        
        for sample_type in ABNV_ORDER:
            type_data = conc_data[conc_data['SampleType'] == sample_type]
            if len(type_data) > 0:
                X_conc.extend(type_data[feature].values)
                y_conc.extend([sample_type] * len(type_data))
        
        if len(X_conc) >= 6 and len(np.unique(y_conc)) >= 3:
            X_conc = np.array(X_conc).reshape(-1, 1)
            y_conc = np.array(y_conc)
            
            try:
                model = model_config['model'](**model_config['params'])
                y_pred = cross_val_predict(model, X_conc, y_conc, cv=3)
                
                all_true_labels.extend(y_conc)
                all_pred_labels.extend(y_pred)
            except:
                continue
    
    if all_true_labels:
        present_labels = [label for label in ABNV_ORDER if label in np.unique(all_true_labels)]
        cm = confusion_matrix(all_true_labels, all_pred_labels, labels=present_labels)
        accuracy = accuracy_score(all_true_labels, all_pred_labels)
        f1 = f1_score(all_true_labels, all_pred_labels, average='weighted')
        
        return cm, present_labels, accuracy, f1
    else:
        return None, None, 0, 0

## 3. Data Loading and Preprocessing

**Note:** Concentration labeled as 1e3 represents the reference baseline (blank sample with 0.1×PBS only, no analyte).

In [ ]:
# Load impedance data
abnv_data = pd.read_csv('Dataset.csv')

# Verify data structure
print(f"Dataset: {abnv_data.shape[0]} samples × {abnv_data.shape[1]} columns")
print(f"\nSample Types: {sorted(abnv_data['SampleType'].unique())}")
print(f"Frequencies: {sorted(abnv_data['Frequency'].unique())} Hz")
print(f"Concentrations: {sorted(abnv_data['Concentration'].unique())} particles/mL")

# Verify features
available_features = [f for f in IMPEDANCE_FEATURES if f in abnv_data.columns]
available_frequencies = [f for f in FREQUENCIES if f in abnv_data['Frequency'].values]

print(f"\nAvailable Features: {len(available_features)}")
print(f"Available Frequencies: {len(available_frequencies)}")
print(f"Total Combinations: {len(available_features) * len(available_frequencies)}")
print(f"Total Analyses: {len(available_features) * len(available_frequencies) * len(ML_MODELS)}")

## 4. Frequency-Feature Combination Organization

In [ ]:
# Organize data by frequency-feature combinations
frequency_feature_data = {}

for freq in available_frequencies:
    for feature in available_features:
        freq_name = format_frequency_name(freq)
        feature_name = feature.replace('ChangeRate_', '').replace('(%)', '')
        combination_key = f"{freq_name}_{feature_name}"
        
        abnv_combo = abnv_data[abnv_data['Frequency'] == freq]
        
        frequency_feature_data[combination_key] = {
            'frequency': freq,
            'feature': feature,
            'abnv_data': abnv_combo
        }

print(f"Organized {len(frequency_feature_data)} frequency-feature combinations")
print(f"Ready for {len(ML_MODELS)} model evaluations per combination")

## 5. Comprehensive Analysis Pipeline

**Analysis Workflow:**
1. Extract features by AbNV type and concentration
2. Calculate biological scores (linearity and order)
3. Evaluate classification performance for each ML model
4. Compute comprehensive scores
5. Store results for all combinations

In [ ]:
# Initialize results storage
comprehensive_results = {
    'abnv_results': {},
    'optimal_conditions': {},
    'biological_validation': {},
    'comprehensive_scores': {}
}

total_analyses = len(frequency_feature_data) * len(ML_MODELS)
current_analysis = 0

print(f"Starting comprehensive analysis ({total_analyses} total evaluations)...")

# Process each frequency-feature combination
for combination_key, combo_data in frequency_feature_data.items():
    freq = combo_data['frequency']
    feature = combo_data['feature']
    abnv_data_combo = combo_data['abnv_data']
    
    if len(abnv_data_combo) > 0:
        # Exclude reference concentration (1e3) from biological analysis
        analysis_data = abnv_data_combo[abnv_data_combo['Concentration'] > 1e3]
        X_abnv = analysis_data.groupby(['SampleType', 'Concentration'])[feature].mean().reset_index()
        
        # Organize features by type
        features_by_type = {}
        for sample_type in ABNV_ORDER:
            type_data = X_abnv[X_abnv['SampleType'] == sample_type]
            if len(type_data) > 0:
                features_by_type[sample_type] = type_data[feature].values
        
        if len(features_by_type) >= 3:
            # Calculate biological scores
            linearity_score = calculate_linearity_score(features_by_type, ABNV_ORDER)
            order_score = calculate_order_score(features_by_type, ABNV_ORDER)
            
            combination_model_results = {}
            
            # Evaluate each ML model
            for model_name, model_config in ML_MODELS.items():
                current_analysis += 1
                
                try:
                    classification_score, true_labels, pred_labels, acc_std, f1_std = \
                        calculate_concentration_aware_classification(analysis_data, feature, model_config)
                    
                    if true_labels and pred_labels:
                        accuracy = accuracy_score(true_labels, pred_labels)
                        f1 = f1_score(true_labels, pred_labels, average='weighted')
                        precision = precision_score(true_labels, pred_labels, average='weighted')
                        recall = recall_score(true_labels, pred_labels, average='weighted')
                    else:
                        accuracy = f1 = precision = recall = 0
                    
                    comprehensive_score = calculate_comprehensive_score(
                        linearity_score, order_score, classification_score
                    )
                    
                    combination_model_results[model_name] = {
                        'accuracy': accuracy,
                        'f1_score': f1,
                        'precision': precision,
                        'recall': recall,
                        'classification_score': classification_score,
                        'linearity_score': linearity_score,
                        'order_score': order_score,
                        'comprehensive_score': comprehensive_score,
                        'cv_std': {
                            'accuracy': acc_std,
                            'f1': f1_std
                        }
                    }
                except:
                    combination_model_results[model_name] = {
                        'accuracy': 0, 'f1_score': 0, 'precision': 0, 'recall': 0,
                        'classification_score': 0, 'linearity_score': linearity_score,
                        'order_score': order_score, 'comprehensive_score': 0,
                        'cv_std': {'accuracy': 0, 'f1': 0}
                    }
            
            # Store results
            comprehensive_results['abnv_results'][combination_key] = {
                'frequency': freq,
                'feature': feature,
                'linearity_score': linearity_score,
                'order_score': order_score,
                'model_results': combination_model_results,
                'data_info': {
                    'abnv_samples': len(abnv_data_combo),
                    'abnv_classes': len(features_by_type),
                    'class_distribution': {st: len(samples) for st, samples in features_by_type.items()}
                }
            }

print(f"\nAnalysis complete: {len(comprehensive_results['abnv_results'])} combinations processed")
print(f"Total evaluations: {current_analysis}")

## 6. Biological Order Validation

Validation of expected biological order: NC (0) < Monomer (1) < Trimer (3) < Nonamer (9)

In [ ]:
# Extract biological validation data
biological_validation_data = []

for combo_key, combo_results in comprehensive_results['abnv_results'].items():
    freq = combo_results['frequency']
    feature = combo_results['feature']
    linearity_score = combo_results['linearity_score']
    order_score = combo_results.get('order_score', 0)
    
    combo_data = frequency_feature_data[combo_key]
    abnv_data_combo = combo_data['abnv_data']
    
    if len(abnv_data_combo) > 0:
        analysis_data = abnv_data_combo[abnv_data_combo['Concentration'] > 1e3]
        X_abnv = analysis_data.groupby(['SampleType', 'Concentration'])[feature].mean().reset_index()
        
        type_means = {}
        for sample_type in ABNV_ORDER:
            type_data = X_abnv[X_abnv['SampleType'] == sample_type]
            if len(type_data) > 0:
                type_means[sample_type] = type_data[feature].mean()
        
        if len(type_means) >= 3:
            biological_validation_data.append({
                'combination': combo_key,
                'frequency': freq,
                'feature': feature,
                'linearity_score': linearity_score,
                'order_score': order_score,
                'type_means': type_means,
                'freq_name': format_frequency_name(freq),
                'feature_name': feature.replace('ChangeRate_', '').replace('(%)', '')
            })

print(f"Biological validation data: {len(biological_validation_data)} combinations")

In [ ]:
# Biological validation visualization
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# 1. Linearity score by feature
ax1 = axes[0, 0]
features = ['Z', 'Real', 'Imag']
feature_linearity = {feat: [] for feat in features}

for data in biological_validation_data:
    feat_name = data['feature_name']
    if feat_name in feature_linearity:
        feature_linearity[feat_name].append(data['linearity_score'])

box_data = [scores for scores in feature_linearity.values() if scores]
box_labels = [feat for feat, scores in feature_linearity.items() if scores]

bp1 = ax1.boxplot(box_data, labels=box_labels, patch_artist=True)
colors = [FEATURE_COLORS[f] for f in box_labels]
for patch, color in zip(bp1['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax1.set_title('Linearity Score Distribution by Feature', fontsize=14, fontweight='bold')
ax1.set_ylabel('Linearity Score (R²)', fontsize=12)
ax1.grid(True, alpha=0.3)

# 2. Top 3 combinations biological order pattern
ax2 = axes[0, 1]
top_3_linearity = sorted(biological_validation_data, key=lambda x: x['linearity_score'], reverse=True)[:3]
biological_orders = [0, 1, 3, 9]
x_positions = np.arange(len(ABNV_ORDER))

for i, data in enumerate(top_3_linearity):
    type_means = data['type_means']
    values = [type_means.get(sample_type, 0) for sample_type in ABNV_ORDER]
    
    ax2.plot(x_positions, values, marker='o', linewidth=2, markersize=6,
            label=f"{data['combination']} ({data['linearity_score']:.3f})", alpha=0.8)

ax2.set_title('Top 3 Combinations: Biological Order Pattern', fontsize=14, fontweight='bold')
ax2.set_xlabel('AbNV Type', fontsize=12)
ax2.set_ylabel('Mean Change Rate (%)', fontsize=12)
ax2.set_xticks(x_positions)
ax2.set_xticklabels([f'{st}\n({order})' for st, order in zip(ABNV_ORDER, biological_orders)])
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

# 3. Linearity vs Order score
ax3 = axes[0, 2]
linearity_scores = [data['linearity_score'] for data in biological_validation_data]
order_scores = [data['order_score'] for data in biological_validation_data]

scatter = ax3.scatter(linearity_scores, order_scores, alpha=0.6, s=60,
                     c=range(len(linearity_scores)), cmap='viridis', edgecolors='black')

ax3.set_xlabel('Linearity Score', fontsize=12, fontweight='bold')
ax3.set_ylabel('Order Score', fontsize=12, fontweight='bold')
ax3.set_title('Linearity vs Order Correlation', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3)

# Add correlation
correlation = np.corrcoef(linearity_scores, order_scores)[0, 1]
ax3.text(0.05, 0.95, f'r = {correlation:.3f}', transform=ax3.transAxes,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8),
        fontsize=11, fontweight='bold')

# 4. Frequency vs Linearity
ax4 = axes[1, 0]
freq_linearity = {}
for data in biological_validation_data:
    freq_name = data['freq_name']
    if freq_name not in freq_linearity:
        freq_linearity[freq_name] = []
    freq_linearity[freq_name].append(data['linearity_score'])

freq_names = list(freq_linearity.keys())
freq_means = [np.mean(scores) for scores in freq_linearity.values()]
freq_stds = [np.std(scores) for scores in freq_linearity.values()]

bars = ax4.bar(freq_names, freq_means, yerr=freq_stds, capsize=5,
              alpha=0.7, color='skyblue', edgecolor='black')
ax4.set_title('Average Linearity by Frequency', fontsize=14, fontweight='bold')
ax4.set_ylabel('Average Linearity Score', fontsize=12)
ax4.tick_params(axis='x', rotation=45)
ax4.grid(True, alpha=0.3, axis='y')

for bar, mean_val in zip(bars, freq_means):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{mean_val:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=9)

# 5. Order correlation heatmap
ax5 = axes[1, 1]
correlation_data = []
combination_labels = []

for data in biological_validation_data[:10]:
    type_means = data['type_means']
    if len(type_means) >= 3:
        values = [type_means.get(st, 0) for st in ABNV_ORDER if st in type_means]
        orders = biological_orders[:len(values)]
        
        if len(values) >= 3:
            correlation, _ = spearmanr(orders, values)
            correlation_data.append(correlation)
            combination_labels.append(data['combination'])

if correlation_data:
    correlation_array = np.array(correlation_data).reshape(-1, 1)
    im = ax5.imshow(correlation_array, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
    
    cbar = plt.colorbar(im, ax=ax5, shrink=0.8)
    cbar.set_label('Spearman Correlation', fontsize=10)
    
    ax5.set_title('Biological Order Correlation', fontsize=14, fontweight='bold')
    ax5.set_yticks(range(len(combination_labels)))
    ax5.set_yticklabels(combination_labels, fontsize=8)
    ax5.set_xticks([0])
    ax5.set_xticklabels(['Correlation'])
    
    for i in range(len(correlation_data)):
        color = 'white' if abs(correlation_data[i]) > 0.5 else 'black'
        ax5.text(0, i, f'{correlation_data[i]:.3f}',
                ha="center", va="center", color=color, fontweight='bold', fontsize=9)

# 6. Summary statistics
ax6 = axes[1, 2]
ax6.axis('off')

avg_linearity = np.mean(linearity_scores)
std_linearity = np.std(linearity_scores)
avg_order = np.mean(order_scores)
std_order = np.std(order_scores)

summary_text = f"""
Biological Order Validation Summary

Total Combinations: {len(biological_validation_data)}

Linearity Score: {avg_linearity:.3f} ± {std_linearity:.3f}
Order Score: {avg_order:.3f} ± {std_order:.3f}

Correlation (r): {correlation:.3f}

Validation Rate:
  Linearity > 0.5: {np.mean([s > 0.5 for s in linearity_scores])*100:.1f}%
  Order > 0.5: {np.mean([s > 0.5 for s in order_scores])*100:.1f}%

Expected Biological Order:
  NC (0) < Monomer (1) < Trimer (3) < Nonamer (9)
"""

ax6.text(0.05, 0.95, summary_text, transform=ax6.transAxes, fontsize=10,
        verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle="round,pad=0.5", facecolor="lightblue", alpha=0.8))

plt.suptitle('Biological Order Validation Analysis', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

## 7. Optimal Conditions Visualization

In [ ]:
# Find best performing combination
best_performance = 0
best_combo_key = None
best_model_name = None

for combo_key, combo_results in comprehensive_results['abnv_results'].items():
    for model_name, model_results in combo_results['model_results'].items():
        if model_results['comprehensive_score'] > best_performance:
            best_performance = model_results['comprehensive_score']
            best_combo_key = combo_key
            best_model_name = model_name

print(f"Optimal Conditions:")
print(f"  Combination: {best_combo_key}")
print(f"  Model: {best_model_name}")
print(f"  Comprehensive Score: {best_performance:.4f}")

In [ ]:
# Optimal conditions comprehensive visualization
best_combo_data = frequency_feature_data[best_combo_key]
best_freq = best_combo_data['frequency']
best_feature = best_combo_data['feature']
best_abnv_data = best_combo_data['abnv_data']

# Exclude reference for analysis
best_analysis_data = best_abnv_data[best_abnv_data['Concentration'] > 1e3]

fig = plt.figure(figsize=(24, 18))
gs = fig.add_gridspec(3, 3, height_ratios=[1.2, 1.2, 0.8], hspace=0.3, wspace=0.3)

# 1. Concentration-Response Curve
ax1 = fig.add_subplot(gs[0, 0:2])
best_analysis_data_copy = best_analysis_data.copy()
best_analysis_data_copy['log_concentration'] = np.log10(best_analysis_data_copy['Concentration'])

for sample_type in ABNV_ORDER:
    type_data = best_analysis_data_copy[best_analysis_data_copy['SampleType'] == sample_type]
    if len(type_data) > 0:
        conc_stats = type_data.groupby('log_concentration')[best_feature].agg(['mean', 'std']).reset_index()
        
        ax1.errorbar(conc_stats['log_concentration'], conc_stats['mean'],
                   yerr=conc_stats['std'], marker='o', markersize=6, linewidth=2, capsize=4,
                   label=f'{sample_type} (n={len(type_data)})',
                   color=ABNV_COLORS.get(sample_type, 'gray'), alpha=0.8)
        
        ax1.scatter(type_data['log_concentration'], type_data[best_feature],
                  color=ABNV_COLORS.get(sample_type, 'gray'), alpha=0.3, s=15)

ax1.set_xlabel('Log₁₀(Concentration)', fontweight='bold', fontsize=12)
ax1.set_ylabel(f'{best_feature.replace("ChangeRate_", "").replace("(%)", "")} (%)',
              fontweight='bold', fontsize=12)
ax1.set_title(f'Concentration-Response Curve\n{format_frequency_name(best_freq)} × {best_feature.replace("ChangeRate_", "").replace("(%)", "")}',
             fontweight='bold', fontsize=14)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# 2. Confusion Matrix
ax2 = fig.add_subplot(gs[1, 0:2])
cm, present_labels, accuracy, f1 = create_confusion_matrix(
    best_analysis_data, best_feature, ML_MODELS[best_model_name]
)

if cm is not None and present_labels is not None:
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues',
               xticklabels=present_labels, yticklabels=present_labels,
               ax=ax2, cbar_kws={'shrink': 0.8}, square=True, annot_kws={'size': 11})
    
    ax2.set_title(f'Confusion Matrix: {best_model_name}', fontweight='bold', fontsize=14)
    ax2.set_xlabel('Predicted', fontweight='bold', fontsize=12)
    ax2.set_ylabel('Actual', fontweight='bold', fontsize=12)
    
    metrics_text = f'Accuracy: {accuracy:.3f}\nF1-Score: {f1:.3f}'
    ax2.text(0.02, 0.98, metrics_text, transform=ax2.transAxes,
           fontsize=11, verticalalignment='top', fontweight='bold',
           bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.9))

# 3. Model Performance Comparison
ax3 = fig.add_subplot(gs[0, 2])
best_combo_results = comprehensive_results['abnv_results'][best_combo_key]
model_performances = []
model_names_list = []

for model_name, model_results in best_combo_results['model_results'].items():
    model_performances.append(model_results['comprehensive_score'])
    model_names_list.append(model_name)

bars = ax3.bar(range(len(model_names_list)), model_performances,
              color=['gold' if name == best_model_name else 'lightblue' for name in model_names_list],
              alpha=0.7, edgecolor='black')

for bar, performance in zip(bars, model_performances):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
           f'{performance:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=9)

ax3.set_xlabel('Models', fontweight='bold', fontsize=11)
ax3.set_ylabel('Comprehensive Score', fontweight='bold', fontsize=11)
ax3.set_title(f'Model Comparison\n{best_combo_key}', fontweight='bold', fontsize=12)
ax3.set_xticks(range(len(model_names_list)))
ax3.set_xticklabels(model_names_list, rotation=45, ha='right', fontsize=9)
ax3.grid(True, alpha=0.3, axis='y')

# 4. Summary
ax4 = fig.add_subplot(gs[1:3, 2])
ax4.axis('off')

all_scores = [model_results['comprehensive_score']
              for combo_results in comprehensive_results['abnv_results'].values()
              for model_results in combo_results['model_results'].values()]

summary_text = f"""
Optimal Conditions Summary

Best Configuration:
  Model: {best_model_name}
  Combination: {best_combo_key}
  Score: {best_performance:.4f}

Frequency: {format_frequency_name(best_freq)}
Feature: {best_feature.replace('ChangeRate_', '').replace('(%)', '')}

Overall Statistics:
  Total combinations: {len(comprehensive_results['abnv_results'])}
  Models per combination: {len(ML_MODELS)}
  
  Average score: {np.mean(all_scores):.3f}
  Score range: {min(all_scores):.3f} - {max(all_scores):.3f}

Analysis Components:
  • Linearity scoring (R²)
  • Biological order validation
  • ML classification
  • Comprehensive metric (weighted)

Biological Order:
  NC(0) < Monomer(1) < Trimer(3) < Nonamer(9)
"""

ax4.text(0.02, 0.95, summary_text, transform=ax4.transAxes, fontsize=10,
        verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle="round,pad=0.5", facecolor="lightcyan", alpha=0.9))

plt.suptitle('Optimal Conditions: Comprehensive Analysis',
            fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

## 8. Multi-Model Performance Heatmap

In [ ]:
# Performance heatmap for all models
features_order = ['Z', 'Real', 'Imag']
frequencies_order = sorted(available_frequencies)
freq_labels = [format_frequency_name(freq) for freq in frequencies_order]

n_models = len(ML_MODELS)
n_cols = 2
n_rows = (n_models + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 6 * n_rows))
if n_rows == 1:
    axes = axes.reshape(1, -1)

for model_idx, model_name in enumerate(ML_MODELS.keys()):
    row = model_idx // n_cols
    col = model_idx % n_cols
    ax = axes[row, col]
    
    # Initialize matrices
    comprehensive_matrix = np.zeros((len(frequencies_order), len(features_order)))
    
    for freq_idx, freq in enumerate(frequencies_order):
        for feat_idx, feature in enumerate(available_features):
            freq_name = format_frequency_name(freq)
            feature_name = feature.replace('ChangeRate_', '').replace('(%)', '')
            combination_key = f"{freq_name}_{feature_name}"
            
            if combination_key in comprehensive_results['abnv_results']:
                combo_results = comprehensive_results['abnv_results'][combination_key]
                if model_name in combo_results['model_results']:
                    model_result = combo_results['model_results'][model_name]
                    feat_matrix_idx = features_order.index(feature_name)
                    comprehensive_matrix[freq_idx, feat_matrix_idx] = model_result['comprehensive_score']
    
    # Create heatmap
    im = ax.imshow(comprehensive_matrix, cmap='RdYlBu_r', aspect='auto',
                  vmin=0, vmax=1, interpolation='nearest')
    
    # Add text annotations
    for freq_idx in range(len(frequencies_order)):
        for feat_idx in range(len(features_order)):
            score = comprehensive_matrix[freq_idx, feat_idx]
            if score > 0:
                text_color = 'white' if score > 0.5 else 'black'
                ax.text(feat_idx, freq_idx, f'{score:.3f}',
                       ha='center', va='center', fontsize=10, fontweight='bold',
                       color=text_color)
    
    # Customize axes
    ax.set_xticks(range(len(features_order)))
    ax.set_xticklabels(features_order, fontsize=12, fontweight='bold')
    ax.set_yticks(range(len(frequencies_order)))
    ax.set_yticklabels(freq_labels, fontsize=12, fontweight='bold')
    
    ax.set_xlabel('Impedance Features', fontsize=13, fontweight='bold')
    ax.set_ylabel('Frequency', fontsize=13, fontweight='bold')
    ax.set_title(f'{model_name}', fontsize=14, fontweight='bold')
    
    # Add grid
    ax.set_xticks(np.arange(-0.5, len(features_order), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(frequencies_order), 1), minor=True)
    ax.grid(which='minor', color='black', linestyle='-', linewidth=1.5, alpha=0.3)
    
    # Highlight best
    best_freq_idx, best_feat_idx = np.unravel_index(np.argmax(comprehensive_matrix),
                                                    comprehensive_matrix.shape)
    if comprehensive_matrix[best_freq_idx, best_feat_idx] > 0:
        rect = plt.Rectangle((best_feat_idx-0.45, best_freq_idx-0.45), 0.9, 0.9,
                           fill=False, edgecolor='gold', linewidth=3)
        ax.add_patch(rect)
    
    # Colorbar
    from mpl_toolkits.axes_grid1 import make_axes_locatable
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    cbar = plt.colorbar(im, cax=cax)
    cbar.set_label('Comprehensive Score', fontsize=11, fontweight='bold')

# Hide empty subplots
for idx in range(n_models, n_rows * n_cols):
    row = idx // n_cols
    col = idx % n_cols
    axes[row, col].set_visible(False)

plt.suptitle('Multi-Model Performance Heatmap: Frequency × Feature Analysis',
            fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

## 9. Results Export

In [ ]:
# Export comprehensive results to CSV
all_results = []

for combo_key, combo_results in comprehensive_results['abnv_results'].items():
    freq = combo_results['frequency']
    feature = combo_results['feature']
    linearity_score = combo_results['linearity_score']
    order_score = combo_results.get('order_score', 0)
    
    freq_name = format_frequency_name(freq)
    feature_name = feature.replace('ChangeRate_', '').replace('(%)', '')
    
    for model_name, model_results in combo_results['model_results'].items():
        row_data = {
            'Combination_Key': combo_key,
            'Frequency_Hz': freq,
            'Frequency_Name': freq_name,
            'Feature': feature,
            'Feature_Name': feature_name,
            'ML_Model': model_name,
            'Accuracy': model_results['accuracy'],
            'F1_Score': model_results['f1_score'],
            'Precision': model_results['precision'],
            'Recall': model_results['recall'],
            'Classification_Score': model_results['classification_score'],
            'Linearity_Score': linearity_score,
            'Biological_Order_Score': order_score,
            'Comprehensive_Score': model_results['comprehensive_score'],
            'Accuracy_Std': model_results['cv_std']['accuracy'],
            'F1_Std': model_results['cv_std']['f1'],
            'AbNV_Samples': combo_results['data_info']['abnv_samples'],
            'AbNV_Classes': combo_results['data_info']['abnv_classes']
        }
        
        class_dist = combo_results['data_info']['class_distribution']
        for class_name in ABNV_ORDER:
            row_data[f'{class_name}_Count'] = class_dist.get(class_name, 0)
        
        all_results.append(row_data)

# Create DataFrame
results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values('Comprehensive_Score', ascending=False)
results_df['Rank'] = range(1, len(results_df) + 1)

# Reorder columns
column_order = [
    'Rank', 'Combination_Key', 'Frequency_Hz', 'Frequency_Name',
    'Feature', 'Feature_Name', 'ML_Model',
    'Comprehensive_Score', 'Accuracy', 'F1_Score', 'Precision', 'Recall',
    'Classification_Score', 'Linearity_Score', 'Biological_Order_Score',
    'Accuracy_Std', 'F1_Std', 'AbNV_Samples', 'AbNV_Classes',
    'NC_Count', 'Monomer_Count', 'Trimer_Count', 'Nonamer_Count'
]

results_df = results_df[column_order]

# Export main results
csv_filename = 'AbNV_Comprehensive_Analysis_Results.csv'
results_df.to_csv(csv_filename, index=False)

print(f"Results exported: {csv_filename}")
print(f"Total records: {len(results_df)}")
print(f"Columns: {len(results_df.columns)}")

In [ ]:
# Summary statistics
print("=" * 80)
print("COMPREHENSIVE ANALYSIS SUMMARY")
print("=" * 80)

print(f"\nDataset Statistics:")
print(f"  Total combinations: {len(results_df['Combination_Key'].unique())}")
print(f"  Models tested: {len(results_df['ML_Model'].unique())}")
print(f"  Total analyses: {len(results_df)}")

print(f"\nPerformance Statistics:")
print(f"  Comprehensive Score Range: {results_df['Comprehensive_Score'].min():.4f} - {results_df['Comprehensive_Score'].max():.4f}")
print(f"  Average Score: {results_df['Comprehensive_Score'].mean():.4f}")
print(f"  Median Score: {results_df['Comprehensive_Score'].median():.4f}")

print(f"\nTop 5 Performing Conditions:")
print("-" * 80)
for _, row in results_df.head(5).iterrows():
    print(f"  Rank {row['Rank']}: {row['Combination_Key']} + {row['ML_Model']}")
    print(f"    Score: {row['Comprehensive_Score']:.4f} | Acc: {row['Accuracy']:.3f} | F1: {row['F1_Score']:.3f}")
    print(f"    Linearity: {row['Linearity_Score']:.3f} | Order: {row['Biological_Order_Score']:.3f}")

In [ ]:
# Model-wise performance summary
model_summary = results_df.groupby('ML_Model').agg({
    'Comprehensive_Score': ['mean', 'std', 'max', 'min'],
    'Accuracy': ['mean', 'std'],
    'F1_Score': ['mean', 'std'],
    'Linearity_Score': 'mean',
    'Biological_Order_Score': 'mean'
}).round(4)

model_summary.columns = ['Score_Mean', 'Score_Std', 'Score_Max', 'Score_Min',
                         'Acc_Mean', 'Acc_Std', 'F1_Mean', 'F1_Std',
                         'Linearity_Mean', 'Order_Mean']

model_summary = model_summary.sort_values('Score_Mean', ascending=False)

# Export model summary
model_summary_filename = 'AbNV_Model_Performance_Summary.csv'
model_summary.to_csv(model_summary_filename)

print(f"\nModel-wise Performance Summary:")
print("-" * 80)
print(model_summary.to_string())
print(f"\nExported: {model_summary_filename}")

In [ ]:
# Frequency-Feature combination summary
combo_summary = results_df.groupby(['Frequency_Name', 'Feature_Name']).agg({
    'Comprehensive_Score': ['mean', 'std', 'max'],
    'Accuracy': 'mean',
    'Linearity_Score': 'mean'
}).round(4)

combo_summary.columns = ['Score_Mean', 'Score_Std', 'Score_Max', 'Acc_Mean', 'Linearity_Mean']
combo_summary = combo_summary.sort_values('Score_Mean', ascending=False)

# Export combination summary
combo_summary_filename = 'AbNV_FrequencyFeature_Performance_Summary.csv'
combo_summary.to_csv(combo_summary_filename)

print(f"\nFrequency-Feature Performance Summary (Top 10):")
print("-" * 80)
print(combo_summary.head(10).to_string())
print(f"\nExported: {combo_summary_filename}")